In [7]:
import os
import pandas as pd
# 1. Data Preparation
if not os.path.exists("input.txt"):
    print("Downloading dataset...")
    # Downloading directly from Hugging Face CSV (no login required)
    url = "https://huggingface.co/datasets/okg/turkish-poems/resolve/main/poems.csv"
    df = pd.read_csv(url)
    
    # Get the column with poems and clean
    # (okg veri setinde şiir metni sütunu 'poem' olarak adlandırılmıştır)
    siirler = df['poem'].dropna().astype(str).tolist()
    
    temiz_metin = ""
    for siir in siirler:
        siir = siir.replace("<br>", "\n").replace("\r", "")
        import re
        siir = re.sub(r"<[^>]+>", "\n", siir)
        temiz_metin += siir + "\n\n\n"
        
    with open("input.txt", "w", encoding="utf-8") as f:
        f.write(temiz_metin)
    print("input.txt successfully created!")
else:
    print("input.txt already exists, skipping download.")
# 2. Read Data
with open("input.txt", 'r', encoding='utf-8') as f:
    text = f.read()
print(f"Total number of characters ready for training: {len(text)}")

input.txt already exists, skipping download.
Total number of characters ready for training: 5334629


In [12]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"engine: {device}")

engine: mps


In [13]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)

stoi = { c:i for i,c in enumerate(chars) }
itos = { i:c for i,c in enumerate(chars) }

encode = lambda s: [stoi[i] for i in s]
decode = lambda n: "".join([itos[i] for i in n])

In [14]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9*len(data))

train_data = data[:n]
val_data = data[n:]

In [15]:
block_size = 128
batch_size = 64

def get_batch(split):
    if split == "train":
        data = train_data
    else:
        data = val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [16]:
xb, yb = get_batch("train")

print("inputs (x) shape:", xb.shape)
print(xb)
print("targets (y) shape:", yb.shape)
print(yb)

inputs (x) shape: torch.Size([64, 128])
tensor([[ 84,   2,  69,  ...,  85,  77,  29],
        [ 65,  75,  65,  ...,  34,  76,   2],
        [ 73,  82,  15,  ...,  77, 110,  68],
        ...,
        [ 15,   2,  59,  ...,   2,  65,  68],
        [ 77, 127,   2,  ...,  68,  73,  76],
        [ 78,  76, 127,  ...,   8,  68,  65]], device='mps:0')
targets (y) shape: torch.Size([64, 128])
tensor([[  2,  69,  76,  ...,  77,  29,  66],
        [ 75,  65,  76,  ...,  76,   2, 129],
        [ 82,  15,  98,  ..., 110,  68,  69],
        ...,
        [  2,  59,  65,  ...,  65,  68,  65],
        [127,   2,  73,  ...,  73,  76,  29],
        [ 76, 127, 125,  ...,  68,  65,  78]], device='mps:0')


In [17]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=6),
            Block(n_embd, n_head=6),
            Block(n_embd, n_head=6),
            Block(n_embd, n_head=6),
            Block(n_embd, n_head=6),
            Block(n_embd, n_head=6),
        )
        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)

        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embd)

        x = tok_emb + pos_emb # (B, T, n_embd)

        x = self.blocks(x)
        x = self.ln_f(x)
        
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [20]:
n_embd = 126

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5) # That C is 3rd dimension of k, so head_size.
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)


class FeedForward(nn.Module):
    
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU(),
        )
    def forward(self, x):
         return self.net(x)

class Block(nn.Module):
    # a tranformer block: attention (communication) + feedforward (thinking)

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head # how many dimensions will each head have? --> ( 32 / 4 = 8)  

        # step_1 --> meeting room (Multi-Head Attention)
        self.sa = MultiHeadAttention(n_head, head_size)

        # step_2 --> thinking room (Feed Forward)
        self.ffwd = FeedForward(n_embd)

        # balancing layers (LayerNorm)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)


    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x 

In [21]:
m = GPTLanguageModel()
m = m.to(device)
optimizer = torch.optim.AdamW(m.parameters(), lr=3e-3)

for step in range(10000):
    xb, yb = get_batch("train")

    logits, loss = m(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Loss after the training:", loss.item())

init = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(init, max_new_tokens=100)[0].tolist()))

Loss after the training: 1.3981218338012695
			Bizim beynim kim bulutsun</p><p>		Gider yüz yaylalar yağar<br/>
	Adına yaralar adam<br/>
		Yaşadık
